In [14]:
import networkx as nx
import pandas as pd

from infomap import Infomap


from sklearn.metrics import normalized_mutual_info_score

## Building the network

In [19]:
# Load edges
edges = pd.read_csv(
    "../data/week4_philosophers_edges.tsv",
    sep="\t",
    comment="#"
)

# Load nodes
nodes = pd.read_csv(
    "../data/week4_philosophers_nodes.tsv",
    sep="\t",
    comment="#"
)

# Make node IDs consistent between the two files
edges["source"] = edges["source"].astype(str)
edges["target"] = edges["target"].astype(str)
nodes["node_id"] = nodes["node_id"].astype(str)

# --------------------------------------------------
# Check consistency between node and edge files
# --------------------------------------------------

node_ids = set(nodes["node_id"])
edge_nodes = set(edges["source"]) | set(edges["target"])

# Nodes referenced by edges but missing from nodes file
missing_from_nodes = edge_nodes - node_ids

# Nodes in nodes file but appearing in no edge
isolated_nodes = node_ids - edge_nodes

print("Nodes file:", len(nodes))
print("Edges file:", len(edges))

print(
    "Nodes appearing in edges but missing from nodes file:",
    len(missing_from_nodes)
)

print(
    "Nodes in nodes file but appearing in no edges:",
    len(isolated_nodes)
)

# --------------------------------------------------
# Combine duplicate directed edges
# --------------------------------------------------

edges = (
    edges
    .groupby(["source", "target"], as_index=False)["weight"]
    .sum()
)

# --------------------------------------------------
# Create directed weighted graph
# --------------------------------------------------

G = nx.from_pandas_edgelist(
    edges,
    source="source",
    target="target",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

# --------------------------------------------------
# Add all nodes from nodes file
# This also preserves isolated nodes
# --------------------------------------------------

node_attributes = nodes.set_index("node_id").to_dict("index")

G.add_nodes_from(
    (node_id, attributes)
    for node_id, attributes in node_attributes.items()
)

# --------------------------------------------------
# Network summary
# --------------------------------------------------

print("\nNetwork:")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Total weight:", G.size(weight="weight"))
print("Directed:", G.is_directed())

Nodes file: 1444
Edges file: 11135
Nodes appearing in edges but missing from nodes file: 0
Nodes in nodes file but appearing in no edges: 68

Network:
Nodes: 1444
Edges: 11135
Total weight: 15758.0
Directed: True


### Louvain

To assess the stability of the Louvain solution, we run the algorithm multiple times with different random seeds and compare the resulting modularity values and community structures

In [20]:
# --------------------------------------------------
# Run Louvain multiple times
# --------------------------------------------------

n_runs = 20
louvain_results = []

for seed in range(n_runs):

    communities = nx.community.louvain_communities(
        G,
        weight="weight",
        seed=seed
    )

    # Calculate modularity
    Q = nx.community.modularity(
        G,
        communities,
        weight="weight"
    )

    louvain_results.append({
        "seed": seed,
        "modularity": Q,
        "n_communities": len(communities),
        "communities": communities
    })

print(f"Completed {n_runs} Louvain runs.")

Completed 20 Louvain runs.


In [21]:
# --------------------------------------------------
# Compare Louvain runs
# --------------------------------------------------

for result in louvain_results:
    print(
        f"Seed {result['seed']:2d} | "
        f"Modularity = {result['modularity']:.6f} | "
        f"Communities = {result['n_communities']}"
    )

Seed  0 | Modularity = 0.556229 | Communities = 77
Seed  1 | Modularity = 0.566680 | Communities = 78
Seed  2 | Modularity = 0.551174 | Communities = 78
Seed  3 | Modularity = 0.563586 | Communities = 78
Seed  4 | Modularity = 0.559677 | Communities = 78
Seed  5 | Modularity = 0.556170 | Communities = 77
Seed  6 | Modularity = 0.557484 | Communities = 77
Seed  7 | Modularity = 0.563936 | Communities = 79
Seed  8 | Modularity = 0.567563 | Communities = 78
Seed  9 | Modularity = 0.563717 | Communities = 79
Seed 10 | Modularity = 0.563640 | Communities = 78
Seed 11 | Modularity = 0.558860 | Communities = 78
Seed 12 | Modularity = 0.560886 | Communities = 77
Seed 13 | Modularity = 0.555092 | Communities = 77
Seed 14 | Modularity = 0.555769 | Communities = 78
Seed 15 | Modularity = 0.556784 | Communities = 77
Seed 16 | Modularity = 0.557798 | Communities = 78
Seed 17 | Modularity = 0.558825 | Communities = 78
Seed 18 | Modularity = 0.563645 | Communities = 77
Seed 19 | Modularity = 0.561220

In [22]:
modularities = [r["modularity"] for r in louvain_results]
community_counts = [r["n_communities"] for r in louvain_results]

print("\nModularity range:")
print(f"Minimum: {min(modularities):.6f}")
print(f"Maximum: {max(modularities):.6f}")

print("\nNumber of communities:")
print(f"Minimum: {min(community_counts)}")
print(f"Maximum: {max(community_counts)}")


Modularity range:
Minimum: 0.551174
Maximum: 0.567563

Number of communities:
Minimum: 77
Maximum: 79


For the number of communities across these 20 runs louvain is mostly stable

In [23]:
# --------------------------------------------------
# Select the Louvain partition with highest modularity
# --------------------------------------------------

best_louvain = max(
    louvain_results,
    key=lambda x: x["modularity"]
)

communities_louvain = best_louvain["communities"]

print("Best Louvain result")
print("----------------------------")
print("Seed:", best_louvain["seed"])
print("Modularity:", best_louvain["modularity"])
print("Number of communities:", best_louvain["n_communities"])

Best Louvain result
----------------------------
Seed: 8
Modularity: 0.5675628433940749
Number of communities: 78


In [25]:
# Convert communities into node -> community label

partition_louvain = {}

for community_id, community in enumerate(communities_louvain):
    for node in community:
        partition_louvain[node] = community_id

print("\nNodes assigned:", len(partition_louvain))
print("Nodes in network:", G.number_of_nodes())
# Community sizes for the best Louvain partition

louvain_sizes = sorted(
    [len(c) for c in communities_louvain],
    reverse=True
)

for i, size in enumerate(louvain_sizes, start=1):
    print(f"Community {i}: {size} nodes")


Nodes assigned: 1444
Nodes in network: 1444
Community 1: 280 nodes
Community 2: 248 nodes
Community 3: 204 nodes
Community 4: 188 nodes
Community 5: 154 nodes
Community 6: 128 nodes
Community 7: 95 nodes
Community 8: 75 nodes
Community 9: 2 nodes
Community 10: 2 nodes
Community 11: 1 nodes
Community 12: 1 nodes
Community 13: 1 nodes
Community 14: 1 nodes
Community 15: 1 nodes
Community 16: 1 nodes
Community 17: 1 nodes
Community 18: 1 nodes
Community 19: 1 nodes
Community 20: 1 nodes
Community 21: 1 nodes
Community 22: 1 nodes
Community 23: 1 nodes
Community 24: 1 nodes
Community 25: 1 nodes
Community 26: 1 nodes
Community 27: 1 nodes
Community 28: 1 nodes
Community 29: 1 nodes
Community 30: 1 nodes
Community 31: 1 nodes
Community 32: 1 nodes
Community 33: 1 nodes
Community 34: 1 nodes
Community 35: 1 nodes
Community 36: 1 nodes
Community 37: 1 nodes
Community 38: 1 nodes
Community 39: 1 nodes
Community 40: 1 nodes
Community 41: 1 nodes
Community 42: 1 nodes
Community 43: 1 nodes
Comm

Louvain identified 78 communities in total. However, 68 of these are singleton communities corresponding exactly to the 68 isolated nodes in the network. Among the 1,376 connected nodes, Louvain identifies 10 substantive communities.

## Info Map

In [42]:
from infomap import Infomap

n_infomap_runs = 20
infomap_results = []

for seed in range(1, n_infomap_runs + 1):

    im = Infomap(
        directed=True,
        seed=seed
    )

    # Add directed weighted edges
    for u, v, data in G.edges(data=True):
        im.add_link(
            node_to_int[u],
            node_to_int[v],
            float(data["weight"])
        )

    im.run()

    # Extract partition
    partition = {}

    for node in im.tree:
        if node.is_leaf:
            original_node = int_to_node[node.node_id]
            partition[original_node] = node.module_id

    infomap_results.append({
        "seed": seed,
        "partition": partition,
        "n_communities": len(set(partition.values())),
        "codelength": im.codelength
    })

print(f"Completed {n_infomap_runs} Infomap runs.")

Completed 20 Infomap runs.


In [44]:
for result in infomap_results:
    print(
        f"Seed {result['seed']:2d} | "
        f"Communities = {result['n_communities']:2d} | "
        f"Codelength = {result['codelength']:.6f}"
    )

Seed  1 | Communities =  5 | Codelength = 7.533557
Seed  2 | Communities =  4 | Codelength = 7.529865
Seed  3 | Communities =  4 | Codelength = 7.542987
Seed  4 | Communities =  4 | Codelength = 7.545115
Seed  5 | Communities =  5 | Codelength = 7.527683
Seed  6 | Communities =  5 | Codelength = 7.542920
Seed  7 | Communities =  4 | Codelength = 7.557553
Seed  8 | Communities =  5 | Codelength = 7.531637
Seed  9 | Communities =  4 | Codelength = 7.540135
Seed 10 | Communities =  4 | Codelength = 7.529476
Seed 11 | Communities =  5 | Codelength = 7.540053
Seed 12 | Communities =  5 | Codelength = 7.566768
Seed 13 | Communities =  5 | Codelength = 7.544443
Seed 14 | Communities =  5 | Codelength = 7.539995
Seed 15 | Communities =  4 | Codelength = 7.534621
Seed 16 | Communities =  5 | Codelength = 7.534256
Seed 17 | Communities =  4 | Codelength = 7.549295
Seed 18 | Communities =  4 | Codelength = 7.539513
Seed 19 | Communities =  4 | Codelength = 7.533340
Seed 20 | Communities =  5 | Co

In [45]:
best_infomap = min(
    infomap_results,
    key=lambda x: x["codelength"]
)

partition_infomap = best_infomap["partition"]

print("Best Infomap result")
print("----------------------------")
print("Seed:", best_infomap["seed"])
print("Codelength:", best_infomap["codelength"])
print("Number of communities:", best_infomap["n_communities"])
print("Nodes assigned:", len(partition_infomap))

Best Infomap result
----------------------------
Seed: 5
Codelength: 7.5276827561411235
Number of communities: 5
Nodes assigned: 1376


In [46]:
# --------------------------------------------------
# Analyze the best Infomap partition
# --------------------------------------------------

partition_infomap = best_infomap["partition"]

# Nodes assigned to Infomap
infomap_nodes = set(partition_infomap.keys())

# Isolated nodes in the original network
network_nodes = set(G.nodes())
isolated_nodes = network_nodes - set(
    edges["source"]
).union(set(edges["target"]))

# Check that Infomap's missing nodes are exactly the isolated nodes
missing_from_infomap = network_nodes - infomap_nodes

print("INFOMAP PARTITION")
print("----------------------------")
print("Seed:", best_infomap["seed"])
print("Codelength:", best_infomap["codelength"])
print("Total nodes in network:", G.number_of_nodes())
print("Nodes assigned to Infomap:", len(infomap_nodes))
print("Nodes not assigned:", len(missing_from_infomap))

print(
    "Missing nodes are exactly the isolated nodes:",
    missing_from_infomap == isolated_nodes
)

# --------------------------------------------------
# Community sizes
# --------------------------------------------------

community_sizes = (
    pd.Series(partition_infomap)
    .value_counts()
    .sort_index()
)

print("\nCommunity sizes:")
for community, size in community_sizes.items():
    percentage = size / len(infomap_nodes) * 100
    print(
        f"Community {community}: "
        f"{size} nodes ({percentage:.2f}%)"
    )

# --------------------------------------------------
# Isolated nodes
# --------------------------------------------------

print("\nIsolated nodes:", len(isolated_nodes))
print(
    "Total check:",
    len(infomap_nodes) + len(isolated_nodes),
    "=",
    G.number_of_nodes()
)

INFOMAP PARTITION
----------------------------
Seed: 5
Codelength: 7.5276827561411235
Total nodes in network: 1444
Nodes assigned to Infomap: 1376
Nodes not assigned: 68
Missing nodes are exactly the isolated nodes: True

Community sizes:
Community 1: 1278 nodes (92.88%)
Community 2: 89 nodes (6.47%)
Community 3: 6 nodes (0.44%)
Community 4: 2 nodes (0.15%)
Community 5: 1 nodes (0.07%)

Isolated nodes: 68
Total check: 1444 = 1444


why is infomap separting one philosopher in one community? 

In [47]:
# Show the 5-community solution, including the singleton
for community, size in community_sizes.items():

    members = [
        node for node, comm in partition_infomap.items()
        if comm == community
    ]

    print(f"\nCommunity {community} ({len(members)} nodes):")

    for node in members:
        name = G.nodes[node].get("name", node)
        print("  ", name)


Community 1 (1278 nodes):
   Aristotle
   Plato
   Diogenes Laertius
   Socrates
   Plutarch
   Pythagoras
   Galen
   Porphyry of Tyre
   Plotinus
   Theophrastus
   Heraclitus
   Simplicius of Cilicia
   Empedocles
   Euclid
   Parmenides
   Proclus
   Archimedes
   Boethius
   Archytas
   Michel de Montaigne
   John Philoponus
   Democritus
   Aristoxenus
   Iamblichus
   Sextus Empiricus
   Xenophon
   Speusippus
   Eudoxus of Cnidus
   Jean Buridan
   Seneca the Younger
   Anaxagoras
   Marsilio Ficino
   Euclid of Megara
   Hippocrates
   Pyrrho
   Alexander of Aphrodisias
   Alcibiades
   Lucian
   Ammonius Hermiae
   Zeno of Elea
   Zeno of Citium
   Aristippus
   Chrysippus
   Xenocrates
   Philolaus
   Cratylus
   Paracelsus
   Damascius
   Xenophanes
   Photios I of Constantinople
   Carneades
   Favorinus
   Marcus Aurelius
   Apuleius
   Syrianus
   Olympiodorus the Younger
   William of Moerbeke
   Antisthenes
   Diogenes
   Ammonius Saccas
   Anaximander
   Timon of Phl

In [48]:
# Find Fritz Mauthner's node ID
fritz_nodes = [
    node for node, data in G.nodes(data=True)
    if data.get("name") == "Fritz Mauthner"
]

print(fritz_nodes)

['Fritz_Mauthner']


In [49]:
fritz = fritz_nodes[0]

print("Fritz Mauthner")
print("----------------")
print("Node ID:", fritz)
print("Louvain community:", partition_louvain[fritz])
print("Infomap community:", partition_infomap[fritz])

print("\nLouvain community members:")

louvain_comm = partition_louvain[fritz]

for node, comm in partition_louvain.items():
    if comm == louvain_comm:
        print(G.nodes[node].get("name", node))

Fritz Mauthner
----------------
Node ID: Fritz_Mauthner
Louvain community: 2
Infomap community: 5

Louvain community members:
Edgar S. Brightman
Henry George
Lorenz Oken
Harald Høffding
Friedrich Engels
Stanisław Leśniewski
Leon Trotsky
Immanuel Kant
Johann Georg Hamann
Franz Rosenzweig
Johann Friedrich Herbart
Baruch Spinoza
Albert Schweitzer
George Stout
Georg Christoph Lichtenberg
Georgi Plekhanov
Karl Wilhelm Ferdinand Solger
Wesley Newcomb Hohfeld
Johannes Nikolaus Tetens
Giovanni Gentile
Max Müller
Tetsurō Watsuji
Bernard Bolzano
Leonard Nelson
Lewis Carroll
Norman Kemp Smith
Wilhelm Wundt
Israel Salanter
Robert Adamson (philosopher)
Emil du Bois-Reymond
Hans Freyer
Carl Jung
Martin Heidegger
Shūzō Kuki
Vincenzo Gioberti
Julius Evola
Jerome Frank
Béla Hamvas
Vladimir Lenin
Max Weber
Alexius Meinong
Friedrich Carl von Savigny
Hryhorii Skovoroda
Reinhold Niebuhr
Rudolf Haym
Karl Rosenkranz
Christian von Ehrenfels
Rudolf Christoph Eucken
Salomon Maimon
Jakob Sigismund Beck
Leo Strau

In [50]:
best_4 = min(
    [r for r in infomap_results if r["n_communities"] == 4],
    key=lambda x: x["codelength"]
)

print("Best 4-community solution")
print("----------------------------")
print("Seed:", best_4["seed"])
print("Codelength:", best_4["codelength"])

Best 4-community solution
----------------------------
Seed: 10
Codelength: 7.529476319155729


In [51]:
print("\nComparison")
print("----------------------------")
print(
    "Best 4-community codelength:",
    best_4["codelength"]
)
print(
    "Best 5-community codelength:",
    best_infomap["codelength"]
)
print(
    "Difference:",
    best_4["codelength"] - best_infomap["codelength"]
)


Comparison
----------------------------
Best 4-community codelength: 7.529476319155729
Best 5-community codelength: 7.5276827561411235
Difference: 0.001793563014605759


Infomap produced either four or five communities across the different runs. The best five-community solution had a codelength of 7.528, only slightly lower than the best four-community solution (7.529). We therefore retained the five-community solution, including its singleton community containing Fritz Mauthner, rather than removing it manually. This allows us to examine a particularly clear disagreement between Infomap and Louvain.

## NMI

Louvain includes the 68 isolated nodes as singleton communities, while Infomap does not assign those nodes to flow communities. Therefore, for a fair NMI comparison, we calculated NMI on the 1,376 connected nodes that both methods partition.

In [52]:
from sklearn.metrics import normalized_mutual_info_score

# Nodes present in both partitions
common_nodes = sorted(
    set(partition_louvain.keys()) &
    set(partition_infomap.keys())
)

print("Nodes in both partitions:", len(common_nodes))

Nodes in both partitions: 1376


In [53]:
# Get the community labels for the same nodes
louvain_labels = [
    partition_louvain[node]
    for node in common_nodes
]

infomap_labels = [
    partition_infomap[node]
    for node in common_nodes
]

# Calculate NMI
nmi = normalized_mutual_info_score(
    louvain_labels,
    infomap_labels
)

print("NMI:", nmi)

NMI: 0.2108431620718549


The NMI between the Louvain and Infomap partitions is 0.211, indicating relatively low agreement between the two community structures.

In [55]:
import pandas as pd

comparison = pd.DataFrame({
    "Louvain": louvain_labels,
    "Infomap": infomap_labels
})

contingency = pd.crosstab(
    comparison["Louvain"],
    comparison["Infomap"]
)

display(contingency)

Infomap,1,2,3,4,5
Louvain,,,,,
0,204,0,0,0,0
1,128,0,0,0,0
2,279,0,0,0,1
3,248,0,0,0,0
4,3,86,6,0,0
5,153,1,0,0,0
6,2,0,0,0,0
7,188,0,0,0,0
8,73,2,0,0,0


The contingency table shows a low level of agreement between the two methods, consistent with the NMI of 0.211. Infomap places the majority of the connected philosophers (1,272 out of 1,376) in a single large community, while Louvain divides these nodes across several different communities. The main exception is Louvain community 4, which is divided by Infomap mainly between communities 2 and 3. Louvain community 9 is also reproduced as the same two-node group by Infomap community 4. In addition, Fritz Mauthner forms a singleton community in Infomap, whereas Louvain places him within its large community 2. Overall, the table indicates that the two methods identify substantially different structures, with Louvain producing a more fine-grained partition and Infomap producing a much coarser, flow-based structure.

In [56]:
# --------------------------------------------------
# Prepare node metadata with community assignments
# --------------------------------------------------

node_info = nodes.copy()

# Add Louvain and Infomap assignments
node_info["Louvain"] = node_info["node_id"].map(partition_louvain)
node_info["Infomap"] = node_info["node_id"].map(partition_infomap)

# Only connected nodes have an Infomap assignment
connected_nodes = node_info[node_info["node_id"].isin(common_nodes)].copy()

print("Connected nodes:", len(connected_nodes))

Connected nodes: 1376


In [57]:
# --------------------------------------------------
# Function to summarize a community
# --------------------------------------------------

def summarize_community(df, method, community_id, n_representatives=10):

    members = df[df[method] == community_id].copy()

    # Dominant era
    era_counts = members["era"].value_counts(dropna=True)

    if len(era_counts) > 0:
        dominant_era = era_counts.index[0]
    else:
        dominant_era = "Unknown"

    # Extract and count subfields
    subfield_counts = {}

    for value in members["subfields"].dropna():

        # Handle common separators
        fields = str(value).replace(";", ",").split(",")

        for field in fields:
            field = field.strip()

            if field:
                subfield_counts[field] = (
                    subfield_counts.get(field, 0) + 1
                )

    dominant_subfields = sorted(
        subfield_counts.items(),
        key=lambda x: x[1],
        reverse=True
    )[:5]

    dominant_subfields_text = ", ".join(
        f"{field} ({count})"
        for field, count in dominant_subfields
    )

    # Representative philosophers
    representatives = members["name"].dropna().head(
        n_representatives
    ).tolist()

    representatives_text = ", ".join(representatives)

    return {
        "Method": method,
        "Community": community_id,
        "Size": len(members),
        "Dominant era": dominant_era,
        "Dominant subfields": dominant_subfields_text,
        "Representative philosophers": representatives_text
    }

In [59]:
# --------------------------------------------------
# Full Louvain community summary
# --------------------------------------------------

louvain_summaries = []

# The 10 non-singleton communities
for community_id in sorted(
    connected_nodes["Louvain"].dropna().unique()
):
    members = connected_nodes[
        connected_nodes["Louvain"] == community_id
    ]

    # Only include substantive communities (> 1 node)
    if len(members) > 1:
        louvain_summaries.append(
            summarize_community(
                connected_nodes,
                "Louvain",
                community_id
            )
        )

louvain_summary = pd.DataFrame(louvain_summaries)

louvain_summary

,Method,Community,Size,Dominant era,Dominant subfields,Representative philosophers
0,Louvain,0,204,centuries BC,"metaphysicians (33), epistemologists (17), log...","Aedesius, Aenesidemus, Agrippa the Skeptic, Al..."
1,Louvain,1,128,11th through 14th centuries,"epistemologists (5), ethicists (4), social and...","'Ali ibn al-'Abbas al-Majusi, Abner of Burgos,..."
2,Louvain,2,280,19th century,"social and political philosophers (36), ethici...","Adolf Reinach, Afrikan Spir, Ahad Ha'am, Alber..."
3,Louvain,3,248,18th century,"social and political philosophers (28), ethici...","A. V. Dicey, Abraham Tucker, Adam Ferguson, Ad..."
4,Louvain,4,95,1st through 10th centuries,"metaphysicians (6), logicians (6), social and ...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S..."
5,Louvain,5,154,19th century,"logicians (18), ethicists (10), social and pol...","Addison Webster Moore, Alain LeRoy Locke, Albe..."
6,Louvain,6,2,19th century,,"George Gurdjieff, P. D. Ouspensky"
7,Louvain,7,188,11th through 14th centuries,"metaphysicians (25), logicians (20), ethicists...","Abbo of Fleury, Abu Ma'shar al-Balkhi, Adam of..."
8,Louvain,8,75,centuries BC,"social and political philosophers (13), ethici...","Cai Yuanpei, Ch'ien Mu, Chao Cuo, Cheng Hao, C..."
9,Louvain,9,2,centuries BC,,"Dirghatamas, Ghosha"


In [60]:
# --------------------------------------------------
# Full Infomap community summary
# --------------------------------------------------

infomap_summaries = []

for community_id in sorted(
    connected_nodes["Infomap"].dropna().unique()
):
    infomap_summaries.append(
        summarize_community(
            connected_nodes,
            "Infomap",
            community_id
        )
    )

infomap_summary = pd.DataFrame(infomap_summaries)

infomap_summary

,Method,Community,Size,Dominant era,Dominant subfields,Representative philosophers
0,Infomap,1.0,1278,19th century,"social and political philosophers (97), metaph...","'Ali ibn al-'Abbas al-Majusi, A. V. Dicey, Abb..."
1,Infomap,2.0,89,1st through 10th centuries,"logicians (7), metaphysicians (6), epistemolog...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S..."
2,Infomap,3.0,6,centuries BC,,"Ashtavakra, Gārgī Vāchaknavī, Maitreyi, Uddāla..."
3,Infomap,4.0,2,centuries BC,,"Dirghatamas, Ghosha"
4,Infomap,5.0,1,19th century,,Fritz Mauthner


To investigate where Louvain and Infomap disagree, we focus on the communities that show splitting, merging, or different assignments in the contingency table. Rather than comparing all communities equally, this analysis identifies the specific philosophical groups whose membership changes between the two methods. We examine the relevant Louvain communities alongside the Infomap communities into which they are divided, as well as cases where the two methods identify the same small group. This allows us to assess whether the differences arise from the different community-detection principles of the two methods and whether they produce different interpretations of the philosopher network.

In [61]:
# --------------------------------------------------
# Targeted analysis of Louvain–Infomap disagreements
# --------------------------------------------------

# Start from the contingency table we already calculated
contingency = pd.crosstab(
    comparison["Louvain"],
    comparison["Infomap"]
)

disagreement_rows = []

for louvain_comm in contingency.index:

    row = contingency.loc[louvain_comm]

    # Total nodes in this Louvain community
    total = row.sum()

    # Infomap communities containing members of this
    # Louvain community
    nonzero = row[row > 0]

    # Only consider actual disagreements:
    # more than one Infomap community receives nodes
    if len(nonzero) > 1:

        for infomap_comm, count in nonzero.items():

            percentage = (count / total) * 100

            # Get the philosophers in this subgroup
            subgroup_nodes = [
                node
                for node in common_nodes
                if partition_louvain[node] == louvain_comm
                and partition_infomap[node] == infomap_comm
            ]

            names = [
                G.nodes[node].get("name", node)
                for node in subgroup_nodes
            ]

            # Show up to 10 representatives
            representatives = ", ".join(names[:10])

            disagreement_rows.append({
                "Louvain community": louvain_comm,
                "Louvain size": int(total),
                "Infomap community": infomap_comm,
                "Nodes": int(count),
                "Percentage of Louvain community": round(
                    percentage, 2
                ),
                "Representative philosophers": representatives
            })

targeted_disagreements = pd.DataFrame(
    disagreement_rows
)

targeted_disagreements

,Louvain community,Louvain size,Infomap community,Nodes,Percentage of Louvain community,Representative philosophers
0,2,280,1,279,99.64,"Adolf Reinach, Afrikan Spir, Ahad Ha'am, Alber..."
1,2,280,5,1,0.36,Fritz Mauthner
2,4,95,1,3,3.16,"Ferdinand de Saussure, Shinran, Xiong Shili"
3,4,95,2,86,90.53,"8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S..."
4,4,95,3,6,6.32,"Ashtavakra, Gārgī Vāchaknavī, Maitreyi, Uddāla..."
5,5,154,1,153,99.35,"Addison Webster Moore, Alain LeRoy Locke, Albe..."
6,5,154,2,1,0.65,Emil Leon Post
7,8,75,1,73,97.33,"Cai Yuanpei, Ch'ien Mu, Chao Cuo, Cheng Hao, C..."
8,8,75,2,2,2.67,"Han Yu, Li Ao (philosopher)"


In [62]:
# --------------------------------------------------
# Summary of communities that are split
# --------------------------------------------------

split_summary = []

for louvain_comm in contingency.index:

    row = contingency.loc[louvain_comm]
    nonzero = row[row > 0]

    if len(nonzero) > 1:

        total = row.sum()

        distribution = "; ".join(
            f"Infomap {int(comm)}: {int(count)} "
            f"({count / total * 100:.1f}%)"
            for comm, count in nonzero.items()
        )

        split_summary.append({
            "Louvain community": louvain_comm,
            "Louvain size": int(total),
            "Infomap distribution": distribution
        })

split_summary = pd.DataFrame(split_summary)

split_summary

,Louvain community,Louvain size,Infomap distribution
0,2,280,Infomap 1: 279 (99.6%); Infomap 5: 1 (0.4%)
1,4,95,Infomap 1: 3 (3.2%); Infomap 2: 86 (90.5%); In...
2,5,154,Infomap 1: 153 (99.4%); Infomap 2: 1 (0.6%)
3,8,75,Infomap 1: 73 (97.3%); Infomap 2: 2 (2.7%)


In [63]:
# --------------------------------------------------
# Detailed targeted Louvain–Infomap disagreement table
# --------------------------------------------------

def get_community_characteristics(df, method, community_id):

    members = df[df[method] == community_id].copy()

    # ---------- Dominant era ----------
    era_counts = members["era"].value_counts(dropna=True)

    if len(era_counts) > 0:
        dominant_era = era_counts.index[0]
    else:
        dominant_era = "Unknown"

    # ---------- Dominant subfields ----------
    subfield_counts = {}

    for value in members["subfields"].dropna():

        fields = str(value).replace(";", ",").split(",")

        for field in fields:
            field = field.strip()

            if field:
                subfield_counts[field] = (
                    subfield_counts.get(field, 0) + 1
                )

    dominant_subfields = sorted(
        subfield_counts.items(),
        key=lambda x: x[1],
        reverse=True
    )[:5]

    subfields_text = ", ".join(
        f"{field} ({count})"
        for field, count in dominant_subfields
    )

    # ---------- Representative philosophers ----------
    representatives = (
        members["name"]
        .dropna()
        .head(10)
        .tolist()
    )

    representatives_text = ", ".join(representatives)

    return {
        "Size": len(members),
        "Dominant era": dominant_era,
        "Dominant subfields": subfields_text,
        "Representative philosophers": representatives_text
    }


# --------------------------------------------------
# Identify the main disagreements automatically
# --------------------------------------------------

targeted_rows = []

for louvain_comm in contingency.index:

    row = contingency.loc[louvain_comm]
    nonzero = row[row > 0]

    # Only communities that are split across
    # more than one Infomap community
    if len(nonzero) > 1:

        louvain_total = row.sum()

        # Characteristics of the Louvain community
        louvain_char = get_community_characteristics(
            connected_nodes,
            "Louvain",
            louvain_comm
        )

        for infomap_comm, count in nonzero.items():

            percentage = count / louvain_total * 100

            # Characteristics of the corresponding
            # Infomap subgroup
            infomap_char = get_community_characteristics(
                connected_nodes,
                "Infomap",
                infomap_comm
            )

            targeted_rows.append({
                "Louvain community": louvain_comm,
                "Louvain size": int(louvain_total),
                "Infomap community": int(infomap_comm),
                "Nodes transferred": int(count),
                "Share of Louvain community (%)": round(
                    percentage, 1
                ),

                "Louvain dominant era":
                    louvain_char["Dominant era"],

                "Louvain dominant subfields":
                    louvain_char["Dominant subfields"],

                "Louvain representatives":
                    louvain_char["Representative philosophers"],

                "Infomap dominant era":
                    infomap_char["Dominant era"],

                "Infomap dominant subfields":
                    infomap_char["Dominant subfields"],

                "Infomap representatives":
                    infomap_char["Representative philosophers"]
            })


targeted_disagreements_detailed = pd.DataFrame(
    targeted_rows
)

targeted_disagreements_detailed

,Louvain community,Louvain size,Infomap community,Nodes transferred,Share of Louvain community (%),Louvain dominant era,Louvain dominant subfields,Louvain representatives,Infomap dominant era,Infomap dominant subfields,Infomap representatives
0,2,280,1,279,99.6,19th century,"social and political philosophers (36), ethici...","Adolf Reinach, Afrikan Spir, Ahad Ha'am, Alber...",19th century,"social and political philosophers (97), metaph...","'Ali ibn al-'Abbas al-Majusi, A. V. Dicey, Abb..."
1,2,280,5,1,0.4,19th century,"social and political philosophers (36), ethici...","Adolf Reinach, Afrikan Spir, Ahad Ha'am, Alber...",19th century,,Fritz Mauthner
2,4,95,1,3,3.2,1st through 10th centuries,"metaphysicians (6), logicians (6), social and ...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S...",19th century,"social and political philosophers (97), metaph...","'Ali ibn al-'Abbas al-Majusi, A. V. Dicey, Abb..."
3,4,95,2,86,90.5,1st through 10th centuries,"metaphysicians (6), logicians (6), social and ...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S...",1st through 10th centuries,"logicians (7), metaphysicians (6), epistemolog...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S..."
4,4,95,3,6,6.3,1st through 10th centuries,"metaphysicians (6), logicians (6), social and ...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S...",centuries BC,,"Ashtavakra, Gārgī Vāchaknavī, Maitreyi, Uddāla..."
5,5,154,1,153,99.4,19th century,"logicians (18), ethicists (10), social and pol...","Addison Webster Moore, Alain LeRoy Locke, Albe...",19th century,"social and political philosophers (97), metaph...","'Ali ibn al-'Abbas al-Majusi, A. V. Dicey, Abb..."
6,5,154,2,1,0.6,19th century,"logicians (18), ethicists (10), social and pol...","Addison Webster Moore, Alain LeRoy Locke, Albe...",1st through 10th centuries,"logicians (7), metaphysicians (6), epistemolog...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S..."
7,8,75,1,73,97.3,centuries BC,"social and political philosophers (13), ethici...","Cai Yuanpei, Ch'ien Mu, Chao Cuo, Cheng Hao, C...",19th century,"social and political philosophers (97), metaph...","'Ali ibn al-'Abbas al-Majusi, A. V. Dicey, Abb..."
8,8,75,2,2,2.7,centuries BC,"social and political philosophers (13), ethici...","Cai Yuanpei, Ch'ien Mu, Chao Cuo, Cheng Hao, C...",1st through 10th centuries,"logicians (7), metaphysicians (6), epistemolog...","8th Karmapa, Mikyö Dorje, Abhinavagupta, Adi S..."


The main disagreements between the two methods are concentrated in a small number of communities. Louvain community 4, containing 95 philosophers, is divided by Infomap: 86 philosophers remain together in Infomap community 2, while six are placed in Infomap community 3 and three in the large Infomap community 1. The six philosophers separated into Infomap community 3 are predominantly associated with the centuries BC, whereas the broader Louvain community is dominated by philosophers from the 1st–10th centuries. This suggests that Infomap captures a finer distinction within this part of the network. Other disagreements are smaller, including the isolation of Fritz Mauthner into a one-node Infomap community, while Louvain places him in its large community 2. Overall, these differences illustrate how Louvain's modularity-based approach produces a more fine-grained partition, whereas Infomap's flow-based approach tends to merge much of the network while separating a few structurally distinct groups.

In [65]:
# --------------------------------------------------
# Inspect Infomap Community 3
# --------------------------------------------------

infomap_3_nodes = [
    node for node, comm in partition_infomap.items()
    if comm == 3
]

print("Infomap Community 3")
print("----------------------------")
print("Number of nodes:", len(infomap_3_nodes))

for node in infomap_3_nodes:
    print(
        node,
        "|",
        G.nodes[node].get("name", node),
        "| era:",
        G.nodes[node].get("era", "Unknown")
    )

Infomap Community 3
----------------------------
Number of nodes: 6
Uddālaka_Āruṇi | Uddālaka Āruṇi | era: centuries BC
Yajnavalkya | Yajnavalkya | era: centuries BC
Śvetaketu | Śvetaketu | era: centuries BC
Gārgī_Vāchaknavī | Gārgī Vāchaknavī | era: centuries BC
Maitreyi | Maitreyi | era: centuries BC
Ashtavakra | Ashtavakra | era: centuries BC


In [66]:
# --------------------------------------------------
# Internal edges within Infomap Community 3
# --------------------------------------------------

internal_edges = []

for u, v, data in G.edges(data=True):

    if u in infomap_3_nodes and v in infomap_3_nodes:

        internal_edges.append({
            "source": G.nodes[u].get("name", u),
            "target": G.nodes[v].get("name", v),
            "weight": data["weight"]
        })

internal_edges_df = pd.DataFrame(internal_edges)

print("Internal edges:")
internal_edges_df

Internal edges:


,source,target,weight
0,Ashtavakra,Uddālaka Āruṇi,3
1,Ashtavakra,Śvetaketu,1
2,Uddālaka Āruṇi,Yajnavalkya,4
3,Uddālaka Āruṇi,Śvetaketu,6
4,Śvetaketu,Uddālaka Āruṇi,2
5,Gārgī Vāchaknavī,Maitreyi,1
6,Gārgī Vāchaknavī,Uddālaka Āruṇi,1
7,Gārgī Vāchaknavī,Yajnavalkya,6
8,Maitreyi,Yajnavalkya,2
9,Yajnavalkya,Ashtavakra,1


In [67]:
# --------------------------------------------------
# External connections of Infomap Community 3
# --------------------------------------------------

external_edges = []

community_3_set = set(infomap_3_nodes)

for u, v, data in G.edges(data=True):

    u_in = u in community_3_set
    v_in = v in community_3_set

    # One endpoint inside, one outside
    if u_in != v_in:

        if u_in:
            source = G.nodes[u].get("name", u)
            target = G.nodes[v].get("name", v)
            direction = "outgoing"

        else:
            source = G.nodes[u].get("name", u)
            target = G.nodes[v].get("name", v)
            direction = "incoming"

        external_edges.append({
            "source": source,
            "target": target,
            "weight": data["weight"],
            "direction": direction
        })

external_edges_df = pd.DataFrame(external_edges)

print("Number of external edges:", len(external_edges_df))

print("\nStrongest external connections:")
external_edges_df.sort_values(
    "weight",
    ascending=False
).head(20)

Number of external edges: 14

Strongest external connections:


,source,target,weight,direction
0,Uddālaka Āruṇi,Adi Shankara,1,outgoing
1,Uddālaka Āruṇi,Empedocles,1,outgoing
2,Uddālaka Āruṇi,Madhvacharya,1,outgoing
3,Uddālaka Āruṇi,Max Müller,1,outgoing
4,Uddālaka Āruṇi,Paul Deussen,1,outgoing
5,Uddālaka Āruṇi,Thales of Miletus,1,outgoing
6,Uddālaka Āruṇi,The Buddha,1,outgoing
7,Śvetaketu,Sarvepalli Radhakrishnan,1,outgoing
8,Śvetaketu,Sri Aurobindo,1,outgoing
9,Maitreyi,Adi Shankara,1,outgoing


In [84]:
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------

edges = pd.read_csv(
    "../data/week4_philosophers_edges.tsv",
    sep="\t",
    comment="#"
)

nodes = pd.read_csv(
    "../data/week4_philosophers_nodes.tsv",
    sep="\t",
    comment="#"
)

# Standardise IDs
edges["source"] = edges["source"].astype(str)
edges["target"] = edges["target"].astype(str)
nodes["node_id"] = nodes["node_id"].astype(str)

# Aggregate duplicate directed edges
edges = (
    edges
    .groupby(["source", "target"], as_index=False)["weight"]
    .sum()
)

# Build directed weighted graph
G = nx.from_pandas_edgelist(
    edges,
    source="source",
    target="target",
    edge_attr="weight",
    create_using=nx.DiGraph()
)

# Add all philosophers, including isolates
node_attributes = nodes.set_index("node_id").to_dict("index")

G.add_nodes_from(
    (node_id, attributes)
    for node_id, attributes in node_attributes.items()
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 1444
Edges: 11135


In [85]:
partition_louvain
partition_infomap

{'Aristotle': 1,
 'Plato': 1,
 'Diogenes_Laertius': 1,
 'Socrates': 1,
 'Plutarch': 1,
 'Pythagoras': 1,
 'Galen': 1,
 'Porphyry_of_Tyre': 1,
 'Plotinus': 1,
 'Theophrastus': 1,
 'Heraclitus': 1,
 'Simplicius_of_Cilicia': 1,
 'Empedocles': 1,
 'Euclid': 1,
 'Parmenides': 1,
 'Proclus': 1,
 'Archimedes': 1,
 'Boethius': 1,
 'Archytas': 1,
 'Michel_de_Montaigne': 1,
 'John_Philoponus': 1,
 'Democritus': 1,
 'Aristoxenus': 1,
 'Iamblichus': 1,
 'Sextus_Empiricus': 1,
 'Xenophon': 1,
 'Speusippus': 1,
 'Eudoxus_of_Cnidus': 1,
 'Jean_Buridan': 1,
 'Seneca_the_Younger': 1,
 'Anaxagoras': 1,
 'Marsilio_Ficino': 1,
 'Euclid_of_Megara': 1,
 'Hippocrates': 1,
 'Pyrrho': 1,
 'Alexander_of_Aphrodisias': 1,
 'Alcibiades': 1,
 'Lucian': 1,
 'Ammonius_Hermiae': 1,
 'Zeno_of_Elea': 1,
 'Zeno_of_Citium': 1,
 'Aristippus': 1,
 'Chrysippus': 1,
 'Xenocrates': 1,
 'Philolaus': 1,
 'Cratylus': 1,
 'Paracelsus': 1,
 'Damascius': 1,
 'Xenophanes': 1,
 'Photios_I_of_Constantinople': 1,
 'Carneades': 1,
 'Favo

In [86]:
# ---------------------------------------------------------
# NODE-LEVEL PARTITION DATA
# ---------------------------------------------------------

partition_df = nodes[
    ["node_id", "name", "era", "subfields"]
].copy()

partition_df["Louvain"] = partition_df["node_id"].map(
    partition_louvain
)

partition_df["Infomap"] = partition_df["node_id"].map(
    partition_infomap
)

partition_df.head()

,node_id,name,era,subfields,Louvain,Infomap
0,'Ali_ibn_al-'Abbas_al-Majusi,'Ali ibn al-'Abbas al-Majusi,1st through 10th centuries,NaN,1,1.0
1,"8th_Karmapa,_Mikyö_Dorje","8th Karmapa, Mikyö Dorje",15th and 16th centuries,NaN,4,2.0
2,A._V._Dicey,A. V. Dicey,19th century,NaN,3,1.0
3,Abbo_of_Fleury,Abbo of Fleury,1st through 10th centuries,NaN,7,1.0
4,Abhinavagupta,Abhinavagupta,1st through 10th centuries,aestheticians,4,2.0


In [87]:
# ---------------------------------------------------------
# REMOVE ISOLATED NODES
# ---------------------------------------------------------

non_isolated_nodes = [
    node for node in G.nodes()
    if G.degree(node) > 0
]

G_connected = G.subgraph(
    non_isolated_nodes
).copy()

print(
    "Nodes after removing isolates:",
    G_connected.number_of_nodes()
)

print(
    "Edges:",
    G_connected.number_of_edges()
)

Nodes after removing isolates: 1376
Edges: 11135


In [88]:
# ---------------------------------------------------------
# COMMUNITY CHARACTERISTICS
# ---------------------------------------------------------

def community_characteristics(
    partition_df,
    method,
    community_id
):

    members = partition_df[
        partition_df[method] == community_id
    ].copy()

    size = len(members)

    # -------------------------
    # Dominant era
    # -------------------------

    era_counts = (
        members["era"]
        .dropna()
        .value_counts()
    )

    if len(era_counts) > 0:
        dominant_era = era_counts.index[0]
    else:
        dominant_era = "Unknown"

    # -------------------------
    # Subfields
    # -------------------------

    subfield_counts = {}

    for value in members["subfields"].dropna():

        fields = (
            str(value)
            .replace(";", ",")
            .split(",")
        )

        for field in fields:

            field = field.strip()

            if field:
                subfield_counts[field] = (
                    subfield_counts.get(field, 0) + 1
                )

    dominant_subfields = sorted(
        subfield_counts.items(),
        key=lambda x: x[1],
        reverse=True
    )[:5]

    subfields_text = ", ".join(
        f"{field} ({count})"
        for field, count in dominant_subfields
    )

    # -------------------------
    # Representative philosophers
    # -------------------------

    representatives = (
        members["name"]
        .dropna()
        .head(10)
        .tolist()
    )

    return {
        "size": size,
        "dominant_era": dominant_era,
        "main_subfields": subfields_text,
        "representatives": representatives
    }

In [89]:
# ---------------------------------------------------------
# BUILD COMMUNITY GRAPH
# ---------------------------------------------------------

def build_community_graph(
    G,
    partition,
    partition_df,
    method
):

    # Only nodes with a community assignment
    assigned_nodes = set(partition.keys())

    # Remove isolates / unassigned nodes
    assigned_nodes = {
        node for node in assigned_nodes
        if G.degree(node) > 0
    }

    # ---------------------------------------------
    # Community membership
    # ---------------------------------------------

    community_members = {}

    for node in assigned_nodes:

        community = partition[node]

        if community not in community_members:
            community_members[community] = []

        community_members[community].append(node)

    # ---------------------------------------------
    # Create community graph
    # ---------------------------------------------

    CG = nx.DiGraph()

    # ---------------------------------------------
    # Add community nodes
    # ---------------------------------------------

    for community_id, members in community_members.items():

        characteristics = community_characteristics(
            partition_df,
            method,
            community_id
        )

        CG.add_node(
            community_id,
            size=len(members),
            members=members,
            dominant_era=characteristics["dominant_era"],
            main_subfields=characteristics["main_subfields"],
            representatives=characteristics["representatives"]
        )

    # ---------------------------------------------
    # Aggregate philosopher-level edges
    # ---------------------------------------------

    edge_weights = {}

    for source, target, data in G.edges(data=True):

        if source not in assigned_nodes:
            continue

        if target not in assigned_nodes:
            continue

        source_community = partition[source]
        target_community = partition[target]

        key = (
            source_community,
            target_community
        )

        edge_weights[key] = (
            edge_weights.get(key, 0)
            + data["weight"]
        )

    # ---------------------------------------------
    # Add aggregated edges
    # ---------------------------------------------

    for (source, target), weight in edge_weights.items():

        CG.add_edge(
            source,
            target,
            weight=weight
        )

    return CG

In [90]:
# ---------------------------------------------------------
# LOUVAIN COMMUNITY GRAPH
# ---------------------------------------------------------

CG_louvain = build_community_graph(
    G_connected,
    partition_louvain,
    partition_df,
    "Louvain"
)

print(
    "Louvain:",
    CG_louvain.number_of_nodes(),
    "communities"
)

print(
    "Louvain:",
    CG_louvain.number_of_edges(),
    "community edges"
)

TypeError: '>' not supported between instances of 'DiDegreeView' and 'int'